# 第1章 环境准备

## 本章导读

> 把这一章想象成一张「开工前体检表」就好。你不必在这里掌握 ROCm、HIP、uv 的全部原理——那是后面章节的内容。这一章只做一件事：带你确认三个关键环节。ROCm 能识别 GPU，PyTorch 能调用 GPU，最小 HIP 程序能编译并运行。三个环节都通过，基础链路就打通了；如果某一步未通过，排错指南会帮你尽快定位问题。

本章对应代码在:

```text
code/part0-intro/
├── pyproject.toml
├── uv.lock
├── activate-rocm.sh
└── chapter1/
    ├── check_torch_rocm.py
    └── vector_add.hip
```

## 1.1 本教程的实验基线

本章不打算做"安装百科"，也不会覆盖所有 AMD GPU 和系统组合——真要写成那样，恐怕这一章就得比整本教程还厚。它只回答一个问题：**当前环境，够不够支撑你继续往后学？**

你可以把这一章想成三道门，必须依次推开，一道都跳不过去：

| 门 | 验证什么 | 推开之后说明 |
| ---- | ---- | ---- |
| 第一道门 | ROCm 能看到 GPU | 底层驱动和运行时基本可用 |
| 第二道门 | PyTorch ROCm | 框架层能把计算放到 AMD GPU 上 |
| 第三道门 | 最小 HIP 程序 | 后续手写 kernel 的路径基本打通 |

本章使用的基线如下：

| 项目 | 基线 |
| ---- | ---- |
| 硬件 | AMD Radeon RX 9070 XT |
| GPU 架构 | gfx1201（RDNA4；ISA 名 `gfx12-generic`）|
| ROCm SDK | **10.0.0**（`rocm-sdk version`）|
| HIP 组件版本 | `7.15.26333`（`hipcc --version` / `torch.version.hip`）|
| 操作系统 | **原生 Ubuntu 24.04**（本次验证为 Ubuntu 24.04.5 / Linux 7.0.0-31-generic，x86_64）|
| Python 环境管理 | uv（`uv 0.11.28`）|
| ROCm Python 包来源 | AMD `https://stable.repo.amd.com/rocm/whl-next/` 统一 wheel 源 |

如果你的硬件或 ROCm 版本和上面对不上，不用担心——验证顺序仍然可以照搬，只是包版本、设备名、工具输出会有差异，到时候自己对照一下就好（换卡时的完整调整流程见 [附录 B · 换一张卡](../../appendix/appendix-b-switch-gpu/index.md)）。

本章的安装、GPU 检查与 HIP 编译输出于 **2026-09-19** 在上述实验机复测。注意，**ROCm SDK 与 HIP 组件各有版本号**：装好 ROCm 10.0 后看到 HIP `7.15.26333` 是本次实测的正常输出。

全书各篇的安装环境已统一为 ROCm 10.0。后续章节中标注 ROCm 7.13 的性能表是历史实测，继续保留当时的版本与数值；我们复跑时记录自己的结果即可。

**云平台读者**：预装镜像的软件版本以当前运行时检测为准；它不会因为仓库升级就自动变为 ROCm 10.0。本地安装请使用下文的 uv 环境，云端已有环境可直接继续 GPU 检查。


## 1.2 平台边界：原生 Linux 优先，WSL2 需单独验证

本次 ROCm 10.0 验证使用**原生 Ubuntu 24.04 + RX 9070 XT**。安装 wheel 之前，宿主机需要已经装好支持这张卡的 AMD GPU 驱动，当前用户也需要有访问 GPU 的权限。`uv` 负责项目里的 Python 包和 ROCm 工具链，内核驱动仍由系统管理。

如果我们使用的是 Windows / WSL2，就先按 [AMD 的 ROCm 10.0 安装页](https://rocm.docs.amd.com/en/docs-10.0.0/install/rocm.html)确认显卡、驱动与版本组合，再逐项验证本章的三道门。**本次原生 Linux 的验证结果不能直接当作 WSL2 的验证结果。**

**计算能跑，状态监控也要单独检查**
`rocminfo`、PyTorch 和 HIP 程序通过，只说明对应的计算路径可用。`rocm-smi`、`amd-smi` 与硬件性能计数器还依赖平台提供的驱动接口，不能据此推断它们也能工作。本章以 `rocminfo`、PyTorch 和最小 HIP 程序为必做项；状态监控放在选读区，后续 profiling 以各章实际验证的功能为准。



## 1.3 同步本篇 uv 环境

本地安装从教程仓库根目录开始，使用 `code/part0-intro/pyproject.toml`。默认配置为 **ROCm 10.0 + `device-gfx1201`**；其他架构先按 [附录 B](../../docs/appendix/appendix-b-switch-gpu/index.md)修改三处设备 extras，再执行：

```bash
cd code/part0-intro
uv sync
source ./activate-rocm.sh
```

保留仓库自带的 `uv.lock`，`uv sync` 会在配置变化时更新它，并自动创建或同步 `.venv`。激活脚本会展开 SDK 的开发文件、刷新设备链接。

云平台的预装环境直接继续下一节，以下检查会打印实际版本。


## 1.4 验证 GPU 可见性

**第一道门：确认 ROCm 已经识别 GPU。**

### 检测 GPU 架构

下面的代码优先读取 `HELLO_GPU_ARCH`（只作为 hipcc 编译 target；仅接受 `gfx1100`、`gfx1151`、`gfx1201`）；未设置时，用带 timeout 的 `rocminfo` 精确解析 GPU Agent 的 `Name: gfx...`。如果没有识别到唯一架构，请先检查环境，或填写已经确认过的 `HELLO_GPU_ARCH` 后再继续。

In [ ]:
# 检测当前 GPU 架构（仅用于 hipcc 编译 target）
import os
import re
import subprocess

ARCH_DETECT_TIMEOUT_S = 10
COMPILE_TIMEOUT_S = 120
SMOKE_TIMEOUT_S = 120
FULL_TIMEOUT_S = 300
SUPPORTED_ARCHES = {"gfx1100", "gfx1151", "gfx1201"}
GPU_AGENT_NAME_RE = re.compile(r"(?m)^\s*Name:\s*(gfx[0-9a-z]+)\s*$")
GPU_AGENT_BLOCK_RE = re.compile(
    r"(?ms)^\s*Agent\s+\d+\s*$.*?(?=^\s*Agent\s+\d+\s*$|\Z)"
)


def _gpu_agent_arches(rocminfo_stdout):
    """Return exact gfx names from GPU Agent blocks only."""
    candidates = []
    for block in GPU_AGENT_BLOCK_RE.findall(rocminfo_stdout):
        if re.search(r"(?m)^\s*Device Type:\s*GPU\s*$", block):
            candidates.extend(GPU_AGENT_NAME_RE.findall(block))
    return sorted(set(candidates))


def run_checked(command, *, cwd=None, timeout=SMOKE_TIMEOUT_S):
    """Run a command, expose failures, and stop before stale results are used."""
    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=False,
            cwd=cwd,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired as exc:
        print(f"命令超时（timeout={timeout}s）: {' '.join(command)}")
        if exc.stdout:
            print(f"stdout:\n{exc.stdout}")
        if exc.stderr:
            print(f"stderr:\n{exc.stderr}")
        raise RuntimeError("外部命令超时，已停止后续步骤") from exc
    except FileNotFoundError as exc:
        print(f"命令不存在: {command[0]}")
        raise RuntimeError("外部命令不可用，已停止后续步骤") from exc
    if completed.returncode != 0:
        print(f"命令失败（returncode={completed.returncode}）: {' '.join(command)}")
        if completed.stdout:
            print(f"stdout:\n{completed.stdout}")
        if completed.stderr:
            print(f"stderr:\n{completed.stderr}")
        raise RuntimeError("外部命令失败，已停止后续步骤")
    return completed


override_arch = os.environ.get("HELLO_GPU_ARCH", "").strip()
rocminfo_result = None
if override_arch:
    if override_arch not in SUPPORTED_ARCHES:
        raise ValueError(
            f"HELLO_GPU_ARCH 仅支持 {sorted(SUPPORTED_ARCHES)}；实际值={override_arch!r}"
        )
    arch = override_arch
    arch_source = "override"
else:
    rocminfo_result = run_checked(["rocminfo"], timeout=ARCH_DETECT_TIMEOUT_S)
    gpu_arches = _gpu_agent_arches(rocminfo_result.stdout)
    if len(gpu_arches) != 1:
        raise RuntimeError(
            "rocminfo 必须恰好报告一个 GPU Agent Name: gfx...，"
            f"实际候选={gpu_arches or 'none'}"
        )
    arch = gpu_arches[0]
    arch_source = "rocminfo"

print(f"arch: {arch}")
print(f"arch_source: {arch_source}")
if arch not in SUPPORTED_ARCHES:
    raise RuntimeError(
        f"当前 Chapter 1 不支持检测到的架构 {arch}；支持集合={sorted(SUPPORTED_ARCHES)}"
    )


### 定位仓库根目录

所有路径使用绝对路径，从仓库根目录开始。下面的函数会搜索当前目录及父目录，定位包含 `README.md` + `code/docs/notebooks` 的仓库根目录。

In [ ]:
import pathlib
import subprocess

def find_repo_root():
    """搜索当前目录及父目录，定位包含 README.md + code/docs/notebooks 的仓库根目录。"""
    current = pathlib.Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").is_dir()
            and (candidate / "docs").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError("无法定位仓库根目录")

REPO_ROOT = find_repo_root()
print(f"仓库根目录: {REPO_ROOT}")

### 验证 ROCm 可见性

首次架构检测已经执行并保存了 `rocminfo` 结果。`arch_source=rocminfo` 时，本单元复用该结果并验证候选架构等于 `arch`；`arch_source=override` 时不会再次运行 `rocminfo`，因为 override 只是编译目标，硬件情况请以 PyTorch 与 HIP 小规模检查的运行时结果为准。

In [ ]:
if arch_source == "rocminfo":
    print(rocminfo_result.stdout[:1000])  # 显示首次检测结果的前1000字符
    gpu_arches = _gpu_agent_arches(rocminfo_result.stdout)
    if gpu_arches != [arch]:
        raise RuntimeError(
            "首次 rocminfo 的唯一 GPU Agent 必须等于 arch，"
            f"arch={arch!r}; 实际候选={gpu_arches or 'none'}"
        )
    print(f"\nstatus PASS: ROCm 可见 GPU 设备（GPU Agent: {arch}）")
else:
    print(
        "status NOTE: HELLO_GPU_ARCH override 仅指定 hipcc 编译 target，"
        "硬件情况请以 PyTorch/HIP 小规模检查结果为准。"
    )

### 选读：GPU 状态监控

当前原生 Ubuntu + RX 9070 XT + ROCm 10.0 环境中，`rocm-smi` 和 `amd-smi` 可用于查看功耗、温度、显存用量。其他平台需要单独检查工具支持情况；本章必做项仍是 `rocminfo`、PyTorch 和最小 HIP 程序。


## 1.5 验证 HIP 编译器与 PyTorch ROCm

### 验证 hipcc 编译器

先确认一下 `hipcc` 版本——这个命令顺带还能验证编译器路径是否在 `_rocm_sdk_devel` 下面。

In [ ]:
result = run_checked(["hipcc", "--version"], timeout=SMOKE_TIMEOUT_S)
print(result.stdout)
if result.stderr.strip():
    print(f"编译器 stderr:\n{result.stderr}")
print("status PASS: hipcc 可用")

### 验证 PyTorch ROCm

**第二道门：确认框架层已经把计算放到 AMD GPU 上。**

检查脚本 `check_torch_rocm.py` 做了三件事：导入 PyTorch、检查 GPU 后端、在 GPU 上跑一次 $1024 \\times 1024$ 矩阵乘。流程保持最小，便于快速确认环境；步骤越少，排查时越容易定位问题。

In [ ]:
import sys
import torch

print(f"python_executable: {sys.executable}")
print(f"torch: {torch.__version__}")
print(f"torch.version.hip: {torch.version.hip}")
if not torch.version.hip:
    raise RuntimeError("当前 PyTorch 不是 ROCm 构建：torch.version.hip 为空")

cuda_available = torch.cuda.is_available()
print(f"cuda_available: {cuda_available}")
if not cuda_available:
    raise RuntimeError("PyTorch ROCm 后端不可用")
device_count = torch.cuda.device_count()
if device_count < 1:
    raise RuntimeError("PyTorch 没有可用 GPU device")
device_name = torch.cuda.get_device_name(0)
print(f"device_name: {device_name}")
if not device_name:
    raise RuntimeError("PyTorch GPU device name 为空")

check_torch_script = REPO_ROOT / "code/part0-intro/chapter1/check_torch_rocm.py"
result = run_checked(
    [sys.executable, str(check_torch_script)],
    timeout=SMOKE_TIMEOUT_S,
    cwd=REPO_ROOT / "code/part0-intro",
)
print(result.stdout)
child_device_line = next(
    (line for line in result.stdout.splitlines() if line.startswith("device_name:")),
    "",
)
if not child_device_line.partition(":")[2].strip():
    raise RuntimeError("PyTorch 子脚本缺少非空 device_name")
if "cuda_available: True" not in result.stdout:
    raise RuntimeError("PyTorch 子脚本报告 ROCm 后端不可用")
print("\nstatus PASS: PyTorch ROCm 后端可用")

### 为什么 ROCm 版 PyTorch 里到处都是 `cuda`？

这是一个**几乎每个新手都会问的问题**。答案是历史包袱——PyTorch 的设备字符串一直沿用 `cuda` 这个名字，没有为 ROCm 单独开一条。看到 `device="cuda"` 不代表你跑在 NVIDIA 卡上，把它读作"把 tensor 放到当前可用的 GPU 后端"就行。

这一步过了，至少说明三件事：

| 检查项 | 通过后说明什么 |
| ---- | ---- |
| `import torch` 成功 | Python 环境里的 PyTorch 可用 |
| `cuda_available: True` | PyTorch 能看到 GPU 后端 |
| 矩阵乘完成 | 基础 GPU 计算路径可用 |

## 1.6 验证最小 HIP 程序

**第三道门：你后续手写 kernel 的那条路有没有打通。**

PyTorch 跑通只能证明框架层 OK；要真正走到 GPU 编程，还得能把一段 HIP C++ 编译、加载、跑起来。这里**不要求你立刻理解 HIP 的所有细节**——那是第 2 篇的事情。现在你只需要知道一个最小 HIP 程序的骨架长什么样：

*（图示：最小 HIP 程序的基本路径）*

```text
Host（CPU 侧）准备输入
    ↓
Device（GPU 侧）分配显存
    ↓
Host 到 Device 拷贝（H2D）
    ↓
启动 HIP Kernel
    ↓
Device 到 Host 拷贝（D2H）
    ↓
检查结果 → 释放资源
```

这是几乎所有 HIP / CUDA 程序的最小骨架——分配显存、拷数据、起 kernel、拷回结果、校验、释放。后面写更复杂的算子时，外壳依然是这个样子，变的只是 kernel 内部那几行。

> **💡 第一次看到这些词？**
>
> - **Host（主机）**：CPU 及其使用的系统内存，负责准备输入、发起 GPU 任务和检查结果。示例中的 `h_a`、`h_b`、`h_c` 都是 Host 侧数据。
> - **Device（设备）**：这里指 GPU 及其显存。示例中的 `d_a`、`d_b`、`d_c` 都指向 Device 侧显存；HIP Runtime 和驱动是 Host 程序与 GPU 之间的软件桥梁。
> - **Host 到 Device（H2D）拷贝**：把输入从系统内存传到 GPU 显存；**Device 到 Host（D2H）拷贝**则把计算结果传回系统内存。两者在这里都由 `hipMemcpy` 完成。
> - **HIP Kernel**：在 GPU 上并行执行的计算函数。它只负责计算，**不包含前后的数据拷贝**；本例的 kernel 只执行 `c[idx] = a[idx] + b[idx]`。
>
> Kernel 启动后，Host 不一定会原地等待。示例中的 `hipDeviceSynchronize()` 用来等 GPU 计算完成，再把结果拷回 Host。更复杂的异步执行和同步方式将在后文遇到时展开。

### 编译并运行

下面的代码使用 `hipcc` 编译 `vector_add.hip`，然后运行编译后的可执行文件。编译通过意味着 HIP 编译器认识你的 GPU 架构、能找到对应的 device library。

运行后看到 `status: PASS` 和 `max_error: 0`，三道门全部推开——HIP 编译器认识你的 GPU、kernel 顺利启动、Host 与 Device 之间的数据拷贝正常、结果校验通过。

In [ ]:
import os

chapter1_dir = REPO_ROOT / "code/part0-intro/chapter1"
vector_add_hip = chapter1_dir / "vector_add.hip"
vector_add_bin = chapter1_dir / "vector_add"

# 编译（使用检测到的架构）
compile_result = run_checked(
    ["hipcc", f"--offload-arch={arch}", "-O2", str(vector_add_hip), "-o", str(vector_add_bin)],
    timeout=COMPILE_TIMEOUT_S,
    cwd=chapter1_dir,
)
print("编译成功")
print(compile_result.stderr if compile_result.stderr else "无编译警告")

# 运行；编译失败时 run_checked 已停止，不会消费旧 binary。
run_result = run_checked(
    [str(vector_add_bin)],
    timeout=SMOKE_TIMEOUT_S,
    cwd=chapter1_dir,
)
print("\n运行结果:")
print(run_result.stdout)

if "status: PASS" not in run_result.stdout:
    raise RuntimeError("最小 HIP 程序验证失败")
print("\n✓ 最小 HIP 程序验证通过")

## 1.7 环境不通时先收集什么

环境问题最容易让人焦虑——这一点我们都经历过。但**最糟糕的排错方式是只说一句"跑不通"**。无论求助对象是助教、社区，还是几小时之后冷静下来的你自己，你都需要把模糊的"不行"翻译成别人能判断的具体信息。

一个像样的排错请求，至少需要包含下面这些信息：

| 信息 | 示例 | 为什么重要 |
| ---- | ---- | ---- |
| 机器信息 | Radeon RX 9070 XT（gfx1201）/ ROCm 10.0 / 原生 Ubuntu 24.04 | 明确硬件和软件背景 |
| 目录 | `hello-gpu/code/part0-intro` | 排查路径和环境变量问题 |
| 环境 | 本地：`source ./activate-rocm.sh` 后运行；云端：平台预装 kernel | 判断解释器和环境来源 |
| 命令 | 当前 kernel interpreter：`sys.executable` 所示路径执行 `chapter1/check_torch_rocm.py` | 避免误用 PATH 中的 `python` |
| 期望 | PyTorch 能看到 GPU 并完成矩阵乘 | 明确你认为应该发生什么 |
| 实际 | 完整报错输出 | 保留关键证据 |
| 最近改动 | 刚执行过 `uv sync` | 排查环境变化来源 |

Notebook 中的检查已使用 `sys.executable`；plain-Python 排查时也必须使用当前 kernel interpreter，而不是裸 `python`。先在 Notebook 读取 `sys.executable`，再将该绝对路径代入：

```bash
"/path/printed/by/sys.executable" chapter1/check_torch_rocm.py 2>&1 | tee check_torch_rocm.log
```

最后请把这条铁律刻在心上：**先确认底层，再确认上层**，顺序千万别反过来。

```text
ROCm / GPU 可见性
  ↓
Python 环境
  ↓
PyTorch ROCm
  ↓
HIP / Triton / profiling 工具
```

按这个顺序从下往上排查，无论是自己复盘，还是去问别人，沟通成本都会低很多。

## 附录：环境细节与换卡迁移

本章主线只要求你会跑 `uv sync` 和几个验证命令。下面这些进阶内容已经独立成附录，需要时再翻：

| 你想了解的 | 去哪里看 |
| ---- | ---- |
| 这套环境文件到底是怎么来的？ROCm 10.0 怎样选择设备包和下载源？`rocm-sdk init` 报 "cannot find ROCm device library" 怎么办？ | 附录 A · 环境安装细节与常见坑 |
| 我手上的卡不是 9070 XT（比如 AI MAX 395 / gfx1151），照着本章的 `pyproject.toml` 抄下来 `uv sync` 报错怎么办？ | 附录 B · 换一张卡：切换 ROCm 10.0 的 GPU 架构 |

## 本章小结

- 本章推开了三道环境验证门：**ROCm 可见、PyTorch ROCm、最小 HIP 路径**，每一道都建立在上一道之上。
- 本教程当前基线是原生 Ubuntu 24.04 + ROCm 10.0；其他平台需按显卡、驱动和 SDK 版本单独验证。
- 环境通过 `pyproject.toml` + `uv.lock` 固化，进入 `code/part0-intro` 后只需 `uv sync` 就能复现——不用手动装任何东西。
- `activate-rocm.sh` 负责处理 ROCm wheel 的环境变量，最核心的职责是让 `ROCM_PATH` 指向 `_rocm_sdk_devel`，而不是 `_rocm_sdk_core`。
- PyTorch ROCm 里看到 `cuda:0` 完全正常，是历史命名问题，**不代表**你在用 NVIDIA GPU。
- 环境不通时不要只甩一句"失败了"——把机器信息、目录、命令、完整输出、版本号和最近改动一起拿出来，排错效率会高一个数量级。
- 下一章我们正式进入 GPU 体系结构，把 CU、Wavefront、LDS 这些概念拆开来看——三道门之后的风景，我们来了。

## 延伸阅读

- [uv Documentation](https://docs.astral.sh/uv/)
- [AMD ROCm Documentation](https://rocm.docs.amd.com/)
- [AMD ROCm 10.0 wheel 源](https://stable.repo.amd.com/rocm/whl-next/)
- [PyTorch Get Started](https://pytorch.org/get-started/locally/)